# Annadata — Price forecast → JSON

sklearn only — no TF install.


In [ ]:
!pip install -q scikit-learn
# sklearn already in Colab; pip above is no-op if present
import json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_percentage_error

SEED, OUT = 42, Path("/content/agrosight_exports")
OUT.mkdir(exist_ok=True)
rng = np.random.default_rng(SEED)
COMMODITIES = ["Tomato", "Potato", "Onion", "Wheat", "Soyabean", "Maize"]
BASE = {"Tomato": 1800, "Potato": 1200, "Onion": 1600, "Wheat": 2200, "Soyabean": 4200, "Maize": 1900}

def make_series(name, days=730):
    t = np.arange(days)
    seasonal = 0.12 * np.sin(2 * np.pi * t / 365.25) + 0.05 * np.sin(2 * np.pi * t / 30)
    price = BASE[name] * (1 + seasonal + rng.normal(0, 0.03, size=days))
    return pd.DataFrame({"date": pd.date_range("2024-01-01", periods=days, freq="D"),
                         "commodity": name, "modal_price": price})

df = pd.concat([make_series(c) for c in COMMODITIES], ignore_index=True)
payload = {"model": "GradientBoostingRegressor", "horizon_days": 30, "series": {}}
for crop, g in df.groupby("commodity"):
    y = g.sort_values("date")["modal_price"].astype(float).values
    X, yy = [], []
    for i in range(14, len(y)):
        X.append([y[i-1], y[i-7], y[i-14], i % 365, (i % 365) / 365])
        yy.append(y[i])
    X, yy = np.array(X), np.array(yy)
    split = int(len(X) * 0.8)
    model = GradientBoostingRegressor(random_state=SEED)
    model.fit(X[:split], yy[:split])
    mape = float(mean_absolute_percentage_error(yy[split:], model.predict(X[split:]))) * 100
    hist, forecasts = list(y), []
    for step in range(30):
        i = len(hist)
        feats = np.array([[hist[-1], hist[-7], hist[-14], i % 365, (i % 365) / 365]])
        nxt = float(model.predict(feats)[0]); hist.append(nxt); forecasts.append(round(nxt, 2))
    last = float(y[-1]); change = round((forecasts[-1] - last) / last * 100, 2)
    signal = "WAIT" if change > 2 else ("SELL_NOW" if change < -2 else "HOLD")
    payload["series"][str(crop)] = {
        "last_modal": round(last, 2), "forecast_30d": forecasts,
        "predicted_change_pct": change, "signal": signal, "mape_pct": round(mape, 2),
        "explanation": f"GBR hold-out MAPE {mape:.1f}%. 30d change {change}% → {signal}.",
    }
    print(crop, "MAPE", round(mape, 2), signal)
(OUT / "price_forecast_series.json").write_text(json.dumps(payload, indent=2))
from google.colab import files
files.download(str(OUT / "price_forecast_series.json"))
